# MaxCompute 查询 Notebook（优化版）

本 notebook 用于在 Jupyter 环境中快速查询 MaxCompute 数据并渲染结果

**优化特性：**
- ✅ Instance Tunnel 加速数据传输（与 DataWorks 一致）
- ✅ 显示执行耗时和 LogView 链接
- ✅ SQL 从外部文件读取

In [5]:
# 导入依赖
from odps import ODPS
import pandas as pd
import sys
import os
import time

# 添加父目录到路径，以便导入 config
notebook_dir = os.getcwd()
sys.path.insert(0, notebook_dir)
from config import ODPS_ACCESS_ID, ODPS_ACCESS_KEY, ODPS_PROJECT, ODPS_ENDPOINT

print('✓ 依赖导入成功')

✓ 依赖导入成功


In [6]:
# 建立 MaxCompute 连接
o = ODPS(
    ODPS_ACCESS_ID, ODPS_ACCESS_KEY, ODPS_PROJECT,
    endpoint=ODPS_ENDPOINT
)
print(f'✓ 连接成功: {o.project}')
print(f'  Endpoint: {ODPS_ENDPOINT}')

# 从外部 SQL 文件读取查询语句
sql_file = 'sql/query.sql'
with open(sql_file, 'r', encoding='utf-8') as f:
    sql = f.read().strip()
print(f'✓ 已从 {sql_file} 加载 SQL 查询脚本')

✓ 连接成功: cupshe_bigdata_ads
  Endpoint: https://service.eu-central-1.maxcompute.aliyun.com/api
✓ 已从 sql/query.sql 加载 SQL 查询脚本


### ▶️ 执行查询（Instance Tunnel 加速）

In [7]:
# ========== 执行查询（优化版） ==========
print('正在执行查询...')
start_time = time.time()

# 执行 SQL 并获取 instance
instance = o.execute_sql(sql)

# 显示 Instance ID 和 LogView 链接
print(f'  Instance ID: {instance.id}')
print(f'  LogView: {instance.get_logview_address()}')

# 等待执行完成
instance.wait_for_success()

exec_time = time.time() - start_time
print(f'✓ SQL 执行耗时: {exec_time:.1f}秒')

# 读取结果（使用 tunnel=True 加速数据传输）
print('正在下载结果...')
download_start = time.time()

with instance.open_reader(tunnel=True) as reader:
    df = reader.to_pandas()

download_time = time.time() - download_start
total_time = time.time() - start_time

print(f'✓ 下载耗时: {download_time:.1f}秒')
print(f'✓ 总耗时: {total_time:.1f}秒')
print(f'✓ 查询完成: {len(df)} 行, {len(df.columns)} 列')

# 显示结果
df

正在执行查询...
  Instance ID: 20260701074213824g2j3r2do3ch
  LogView: http://logview.odps.aliyun.com/logview/?h=https://service.eu-central-1.maxcompute.aliyun.com/api&p=cupshe_bigdata_ads&i=20260701074213824g2j3r2do3ch&token=TXFKMkZPQVdueUZYZGJDdkhjeFZ0ZG5yV0RFPSxPRFBTX09CTzpwNF8yMDY0Njg2MTgxMDQ2NzExMTEsMTc4NTQ4Mzc5OCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvY3Vwc2hlX2JpZ2RhdGFfYWRzL2luc3RhbmNlcy8yMDI2MDcwMTA3NDIxMzgyNGcyajNyMmRvM2NoIl19XSwiVmVyc2lvbiI6IjEifQ==
✓ SQL 执行耗时: 70.8秒
正在下载结果...
✓ 下载耗时: 62.0秒
✓ 总耗时: 132.8秒
✓ 查询完成: 75420 行, 12 列


,月,skc,sku,first_published_at,second_category,third_category,category,skc当月在架天数,style_position_name,theme,销量,销售额
0,2026-01,120000205614,130000358126,2025-08-13,连衣裙,连衣裙,成衣,31,明星款,2025Everyday/8 月/8月明星款,2.0,94.745936
1,2026-01,120000205614,130000358126,2025-08-13,针织服装,毛衣裙,成衣,31,明星款,2025Everyday/8 月/8月明星款,2.0,94.745936
2,2026-01,120000205614,130000358127,2025-08-13,连衣裙,连衣裙,成衣,31,明星款,2025Everyday/8 月/8月明星款,NaN,None
3,2026-01,120000205614,130000358127,2025-08-13,针织服装,毛衣裙,成衣,31,明星款,2025Everyday/8 月/8月明星款,NaN,None
4,2026-01,120000205614,130000358128,2025-08-13,连衣裙,连衣裙,成衣,31,明星款,2025Everyday/8 月/8月明星款,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...
75415,2026-05,DAA15B6D001GG,DAA15B6D001GGXL,2026-04-23,睡衣,睡裤,内睡,31,常规款,NaN,1.0,20.27439568
75416,2026-06,DAA15B6D001GG,DAA15B6D001GGL,2026-04-23,睡衣,睡裤,内睡,30,常规款,NaN,NaN,None
75417,2026-06,DAA15B6D001GG,DAA15B6D001GGM,2026-04-23,睡衣,睡裤,内睡,30,常规款,NaN,5.0,85
75418,2026-06,DAA15B6D001GG,DAA15B6D001GGS,2026-04-23,睡衣,睡裤,内睡,30,常规款,NaN,NaN,None


### 💾 导出结果（可选）

In [8]:
# ========== 可选：导出结果 ==========
from datetime import datetime

# 创建输出目录
output_dir = 'notebook_output'
os.makedirs(output_dir, exist_ok=True)

# 生成文件名
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_file = f'{output_dir}/result_{timestamp}.csv'
excel_file = f'{output_dir}/result_{timestamp}.xlsx'

# 保存为 CSV
df.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f'✓ 已保存到 CSV: {csv_file}')

# 保存为 Excel（需要 openpyxl）
try:
    df.to_excel(excel_file, index=False)
    print(f'✓ 已保存到 Excel: {excel_file}')
except ImportError:
    print(' 未安装 openpyxl，跳过 Excel 导出。运行: pip install openpyxl')

✓ 已保存到 CSV: notebook_output/result_20260701_154507.csv
✓ 已保存到 Excel: notebook_output/result_20260701_154507.xlsx
